# Training

Trains the Deep ConvNet (braindecode's `Deep4Net`) for the Table 2 reproduction of Schirrmeister et al. (2017): one model per dataset, frequency band and subject, with the paper's cropped training and two-phase early stopping. Each model is evaluated once on the test set.

Section references are to the [arXiv version](https://arxiv.org/abs/1703.05051v5) of the paper, which includes the supplementary material as Appendix A.

| Step | What | Source |
|---|---|---|
| Data | Preprocessed trials, predefined train/test split | `Preprocessing.ipynb`; A.6 |
| Validation fold | Last part of the training set held out for early stopping | A.4 |
| Model | Deep ConvNet: 4 conv-pool blocks, ELU, max pooling, batch norm, dropout | 2.4.2, Fig. 1, 2.4.4, Table 1 |
| Cropped training | ~2 s crops, predictions for all crops of a trial in one pass | 2.5.4, Fig. 4 |
| Loss | Cross-entropy per crop + tied sample loss | 2.5.1 Eq. 2; 2.5.4 Eq. 3 |
| Optimizer | Adam | 2.5.5 |
| Early stopping | Two phases: stop on validation accuracy, then continue on train + validation | A.4 |
| Trial prediction | Mean of the crop predictions | 2.5.4 |
| Evaluation | Accuracy on the independent test set, mean over subjects | A.4; Table 2 |

The paper does not state batch size, learning rate, validation fold size or the early stopping patience. Those settings are marked **not in the paper** below; our values follow the authors' original braindecode code as far as we know, and should be reported as our choices.

The same code is in `reproduce_table2_deepconvnet.py`, which runs all 46 trainings (e.g. on the HPC) and summarizes them.

In [ ]:
import copy
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from braindecode.models import Deep4Net
from braindecode.util import set_random_seeds

from data_loading import SUBJECTS, load_dataset, split_train_test
from preprocessing import create_trial_windows, preprocess_dataset

mne.set_log_level("ERROR")  # hide MNE's per-channel loading messages
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Settings

Which run to train, and the training settings.

- **Model input size**, 2.5.4: *"the ConvNet input size is also smaller (from about 1000 input samples to about 500 input samples for the 250 Hz sampling rate)"*. The model is built for 1000-sample inputs; with `final_conv_length=2` each prediction then sees 522 samples (≈2.1 s), matching *"we used crops of about 2 seconds as the input"*.
- **Tied loss**, Table 1: the paper's choice is *"Dropout + batch normalization + a new tied loss function"*.
- **Optimizer**, 2.5.5: *"As optimization method, we used Adam"*. **Not in the paper:** learning rate 1e-3 (Adam's default) and batch size 60.
- **Early stopping**, A.4: phase 1 *"stops [...] when validation accuracy does not improve for a predefined number of epochs"*. **Not in the paper:** the size of the validation fold (last 20% of training trials), the number of epochs without improvement (80), the maximum number of epochs (800), and a limit on phase 2 (at most as many epochs as phase 1's best epoch).

Full runs on a CPU take hours per subject. Set `QUICK_TEST = True` to check that everything works with a few epochs.

In [ ]:
DATASET = "BCIC_IV_2a"  # "BCIC_IV_2a" or "HGD"
LOW_CUT_HZ = 4  # 0 or 4, the two frequency ranges of Table 2
SUBJECT = 1
QUICK_TEST = False  # True: only a few epochs, to check that everything runs

N_CLASSES = 4
N_TIMES = 1000  # model input size the model is built for (2.5.4)
FINAL_CONV_LENGTH = 2  # gives crops of 522 samples (~2 s)
TIED_LOSS = True  # Table 1
LR = 1e-3  # not in the paper
BATCH_SIZE = 60  # not in the paper
VALID_FRACTION = 0.2  # not in the paper
MAX_EPOCHS = 800  # not in the paper
MAX_INCREASE_EPOCHS = 80  # not in the paper: phase 1 patience
SEED = 20170629
RESULTS_DIR = Path("results")

if QUICK_TEST:
    MAX_EPOCHS, MAX_INCREASE_EPOCHS = 4, 2

# Table 2: FBCSP accuracy and Deep ConvNet difference to it, in %
TABLE2 = {
    ("BCIC_IV_2a", 0): (68.0, 2.9),
    ("BCIC_IV_2a", 4): (67.8, 2.3),
    ("HGD", 0): (91.2, 1.3),
    ("HGD", 4): (90.9, 0.5),
    ("Combined", 0): (82.1, 1.9),
    ("Combined", 4): (81.9, 1.2),
}

## Data (A.6)

Loads one subject and applies the preprocessing from `Preprocessing.ipynb`. Each trial runs from 0.5 s before the cue to the trial end, 1125 samples at 250 Hz (2.5.3).

The train/test split is the paper's (A.6): for BCIC IV 2a, *"The training set consists of the 288 trials of the first session, the test set of the 288 trials of the second session"*; for HGD, *"The training set consists of the approx. 880 trials of all runs except the last two runs, the test set of the approx. 160 trials of the last 2 runs."*

In [ ]:
def load_trials(name, subject, low_cut_hz):
    """Preprocessed train and test trials as (X, y) arrays in time order."""
    dataset = preprocess_dataset(load_dataset(name, [subject]), name, low_cut_hz)
    train_set, test_set = split_train_test(create_trial_windows(dataset, name))

    def to_arrays(windows):
        X = np.stack([windows[i][0] for i in range(len(windows))])
        y = np.array([windows[i][1] for i in range(len(windows))])
        return X.astype(np.float32), y.astype(np.int64)

    return to_arrays(train_set), to_arrays(test_set)


(X_train, y_train), (X_test, y_test) = load_trials(DATASET, SUBJECT, LOW_CUT_HZ)
print(f"train {X_train.shape}, test {X_test.shape}, train class counts {np.bincount(y_train)}")

## Validation fold (A.4)

*"The early stopping strategy that we use throughout this study [...] splits the training set into a training and validation fold"*.

**Not in the paper:** how large the validation fold is. We hold out the last 20% of the training trials, in time order. The test set is not touched until the very end: *"In our study, all reported accuracies have been determined on an independent test set."*

In [ ]:
n_valid = int(round(len(X_train) * VALID_FRACTION))
to_tensor = lambda a: torch.as_tensor(a, device=device)
X_fit, y_fit = to_tensor(X_train[:-n_valid]), to_tensor(y_train[:-n_valid])
X_valid, y_valid = to_tensor(X_train[-n_valid:]), to_tensor(y_train[-n_valid:])
X_all, y_all = to_tensor(X_train), to_tensor(y_train)  # training + validation fold, for phase 2
X_test_t, y_test_t = to_tensor(X_test), to_tensor(y_test)
print(f"training fold {len(X_fit)}, validation fold {len(X_valid)}, test {len(X_test_t)} trials")

## Model: Deep ConvNet (2.4.2, 2.4.4)

braindecode's `Deep4Net` is the paper's Deep ConvNet, with defaults matching Figure 1:

- *"four convolution-max-pooling blocks, with a special first block designed to handle EEG input [...] followed by three standard convolution-max-pooling blocks and a dense softmax classification layer"* (2.4.2).
- The first block is *"split into a first convolution across time and a second convolution across space (electrodes)"* (2.4.2): 25 temporal filters of length 10, then 25 spatial filters over all electrodes. The later blocks have 50, 100 and 200 filters of length 10; all pooling is max pooling of size 3, stride 3 (Fig. 1).
- *"We used exponential linear units (ELUs [...]) as activation functions"* (2.4.2).
- *"We applied batch normalization [...] to the output of convolutional layers before the nonlinearity"* and *"We drop out the inputs to all convolutional layers after the first with a probability of 0.5"* (2.4.4).

### Cropped training (2.5.4, Fig. 4)

*"we used crops of about 2 seconds as the input"*, one crop per sample (*"which leads to the largest possible number of crops by creating one crop per sample"*). Instead of passing each crop separately, *"we decoded a group of neighboring crops together and reused intermediate convolution outputs"*. `to_dense_prediction_model()` does this (Fig. 4b): the model outputs one prediction per crop for a longer input.

The paper passes inputs of *"about 1000 samples"* (≈500 crops). We pass each whole trial (1125 samples), giving 604 crops per trial. Since `Deep4Net` uses unpadded convolutions, *"this method only results in exactly the same predictions as the naïve method when using valid convolutions (i.e., no padding)"* holds, so the crops and their predictions are the same as in the paper; only the grouping into forward passes differs.

In [ ]:
def build_model(n_chans):
    model = Deep4Net(
        n_chans=n_chans,
        n_outputs=N_CLASSES,
        n_times=N_TIMES,
        final_conv_length=FINAL_CONV_LENGTH,
    )
    model.to_dense_prediction_model()
    return model.to(device)


model = build_model(X_train.shape[1])
n_crops = model(X_fit[:1]).shape[2]
print(model)
print(f"input {X_train.shape[2]} samples -> {n_crops} crop predictions of "
      f"{X_train.shape[2] - n_crops + 1} samples each")

## Loss (2.5.1 Eq. 2, 2.5.4 Eq. 3)

The standard loss is *"the negative log likelihood of the labels"* (Eq. 2), here computed for every crop: *"During training, we compute a loss for each prediction."* Each crop gets the label of its trial (2.5.4).

The tied sample loss: *"In this tied sample loss function, we added the cross-entropy of two neighboring predictions to the usual loss"* (Eq. 3). *"This is meant to make the ConvNet focus on features which are stable for several neighboring input crops."*

The model outputs raw scores, so the softmax of 2.5.1 is applied here.

In [ ]:
def loss_fn(logits, y, tied_loss=TIED_LOSS):
    """Mean per-crop negative log likelihood (Eq. 2) plus the tied sample loss (Eq. 3)."""
    log_probs = F.log_softmax(logits, dim=1)  # (batch, classes, crops)
    targets = y[:, None].expand(-1, log_probs.shape[2])  # every crop gets its trial's label
    loss = F.nll_loss(log_probs, targets)
    if tied_loss:
        # cross-entropy between the predictions of each crop and the next crop
        later_probs = log_probs[:, :, 1:].exp()
        loss = loss - (later_probs * log_probs[:, :, :-1]).sum(dim=1).mean()
    return loss

## Training epoch and evaluation (2.5.1, 2.5.5, 2.5.4)

Parameters are *"optimized via mini-batch stochastic gradient descent using analytical gradients computed via backpropagation"* (2.5.1), with Adam (2.5.5). One epoch is one pass over the trials in random order.

For evaluation, *"The mean of these 625 predictions is used as the final prediction for the trial during the test phase"* (2.5.4); with our trial length that is 604 predictions. We average the softmax probabilities of all crops and take the most likely class.

In [ ]:
def train_epoch(model, optimizer, X, y, rng):
    model.train()
    for batch in np.array_split(rng.permutation(len(X)), max(1, len(X) // BATCH_SIZE)):
        batch = torch.as_tensor(batch, device=X.device)
        optimizer.zero_grad()
        loss = loss_fn(model(X[batch]), y[batch])
        loss.backward()
        optimizer.step()


@torch.no_grad()
def evaluate(model, X, y):
    """Loss and trial accuracy (mean of the crop predictions per trial)."""
    model.eval()
    losses, correct = [], 0
    for start in range(0, len(X), BATCH_SIZE):
        logits = model(X[start:start + BATCH_SIZE])
        yb = y[start:start + BATCH_SIZE]
        losses.append(loss_fn(logits, yb).item() * len(yb))
        trial_probs = F.softmax(logits, dim=1).mean(dim=2)
        correct += (trial_probs.argmax(dim=1) == yb).sum().item()
    return sum(losses) / len(X), correct / len(X)

## Two-phase early stopping (A.4)

The paper uses an early stopping method from the computer vision field ([cuda-convnet methodology](https://web.archive.org/web/20160809230156/https://code.google.com/p/cuda-convnet/wiki/Methodology), A.4 footnote 2):

1. **Phase 1:** *"splits the training set into a training and validation fold and stops the first phase of the training when validation accuracy does not improve for a predefined number of epochs."*
2. **Phase 2:** *"The training continues on the combined training and validation fold starting from the parameter values that led to the best accuracies on the validation fold so far. The training ends when the loss function on the validation fold drops to the same value as the loss function on the training fold at the end of the first training phase"*.

So the number of epochs is not chosen by hand (*"Early stopping in general allows training on different types of networks and datasets without choosing the number of training epochs by hand. Our specific strategy uses the entire training data while only training once."*).

Implementation details:
- "The end of the first training phase" is taken as the best epoch, whose parameters phase 2 starts from. Its training fold loss is the target for phase 2.
- When resetting to the best parameters, the optimizer state is reset too.
- **Not in the paper:** phase 2 stops at the latest after as many epochs as phase 1's best epoch, in case the validation loss never reaches the target.
- The losses compared here are computed in evaluation mode (no dropout, batch norm running statistics), on the full training and validation fold after each epoch.

In [ ]:
def train_two_phase(X_fit, y_fit, X_valid, y_valid, X_all, y_all, verbose=True):
    set_random_seeds(seed=SEED, cuda=device == "cuda")
    rng = np.random.RandomState(SEED)
    model = build_model(X_fit.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    history = []

    def log(phase, epoch):
        train_loss, train_acc = evaluate(model, X_fit, y_fit)
        valid_loss, valid_acc = evaluate(model, X_valid, y_valid)
        history.append(dict(phase=phase, epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                            valid_loss=valid_loss, valid_acc=valid_acc))
        if verbose:
            print(f"phase {phase} epoch {epoch}: train loss {train_loss:.3f} acc {train_acc:.3f} | "
                  f"valid loss {valid_loss:.3f} acc {valid_acc:.3f}", flush=True)
        return history[-1]

    # Phase 1: train on the training fold until validation accuracy stops improving
    best = dict(epoch=0, valid_acc=-1.0)
    for epoch in range(1, MAX_EPOCHS + 1):
        train_epoch(model, optimizer, X_fit, y_fit, rng)
        row = log(1, epoch)
        if row["valid_acc"] > best["valid_acc"]:
            best = dict(row, model=copy.deepcopy(model.state_dict()),
                        optimizer=copy.deepcopy(optimizer.state_dict()))
        if epoch - best["epoch"] >= MAX_INCREASE_EPOCHS:
            break

    # Phase 2: from the best parameters, train on training + validation fold until the
    # validation fold loss drops to the training fold loss at the best epoch
    model.load_state_dict(best["model"])
    optimizer.load_state_dict(best["optimizer"])
    loss_to_reach = best["train_loss"]
    for epoch in range(1, best["epoch"] + 1):
        train_epoch(model, optimizer, X_all, y_all, rng)
        if log(2, epoch)["valid_loss"] < loss_to_reach:
            break

    info = dict(best_epoch=best["epoch"], best_valid_acc=best["valid_acc"], loss_to_reach=loss_to_reach,
                phase1_epochs=sum(h["phase"] == 1 for h in history),
                phase2_epochs=sum(h["phase"] == 2 for h in history))
    return model, pd.DataFrame(history), info

## Train

Trains the subject chosen in the settings. With the full settings this takes hours on a CPU and minutes to an hour on a GPU.

In [ ]:
start = time.time()
model, history, info = train_two_phase(X_fit, y_fit, X_valid, y_valid, X_all, y_all)
train_time_s = time.time() - start
print(info, f"{train_time_s / 60:.1f} min")

## Training curves

Accuracy and loss on the training and validation fold per epoch. The dashed line marks the best phase 1 epoch, where phase 2 starts from; the dotted line in the loss plot is the loss phase 2 has to reach.

In [ ]:
history["step"] = np.arange(1, len(history) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["acc", "loss"]):
    for fold in ["train", "valid"]:
        ax.plot(history["step"], history[f"{fold}_{metric}"], label=f"{fold} fold")
    ax.axvline(info["best_epoch"], color="gray", linestyle="--", label="best phase 1 epoch")
    ax.set_xlabel("epoch (phase 1, then phase 2)")
    ax.set_ylabel(metric)
axes[1].axhline(info["loss_to_reach"], color="gray", linestyle=":", label="phase 2 target loss")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## Test accuracy (A.4)

The test set is used exactly once, after training. Results are saved in the same format as `reproduce_table2_deepconvnet.py`, so both can be summarized together. Quick test runs are not saved.

In [ ]:
test_loss, test_acc = evaluate(model, X_test_t, y_test_t)
print(f"{DATASET}, {LOW_CUT_HZ}-f_end Hz, subject {SUBJECT}: test accuracy {100 * test_acc:.1f}%")

if not QUICK_TEST:
    out_dir = RESULTS_DIR / f"{DATASET}_{LOW_CUT_HZ}Hz"
    out_dir.mkdir(parents=True, exist_ok=True)
    result = dict(dataset=DATASET, low_cut_hz=LOW_CUT_HZ, subject=SUBJECT, test_acc=test_acc,
                  test_loss=test_loss, n_train_trials=len(X_train), n_valid_trials=n_valid,
                  n_test_trials=len(X_test), train_time_s=train_time_s, **info)
    history.drop(columns="step").to_csv(out_dir / f"subject_{SUBJECT:02d}_history.csv", index=False)
    (out_dir / f"subject_{SUBJECT:02d}.json").write_text(json.dumps(result, indent=1))

## Table 2 summary

Table 2 reports the mean test accuracy over subjects per dataset and frequency range, as a difference to FBCSP. The Combined rows are the mean over all subjects of both datasets: e.g. for FBCSP, (9 · 68.0 + 14 · 91.2) / 23 = 82.1.

This collects all saved results, from this notebook and from `reproduce_table2_deepconvnet.py`. The `subjects` column shows how many of the subjects are done.

In [ ]:
rows = [json.loads(p.read_text()) for p in RESULTS_DIR.glob("*/subject_*.json")]
if rows:
    df = pd.DataFrame(rows)
    summary = []
    for low_cut in (0, 4):
        for name in ("BCIC_IV_2a", "HGD", "Combined"):
            names = ("BCIC_IV_2a", "HGD") if name == "Combined" else (name,)
            accs = df[df.dataset.isin(names) & (df.low_cut_hz == low_cut)].test_acc
            fbcsp, paper_delta = TABLE2[(name, low_cut)]
            mean_acc = 100 * accs.mean() if len(accs) else np.nan
            summary.append(dict(
                dataset=name, band=f"{low_cut}-f_end",
                subjects=f"{len(accs)}/{sum(len(SUBJECTS[n]) for n in names)}",
                acc=mean_acc, paper_acc=fbcsp + paper_delta,
                delta_vs_fbcsp=mean_acc - fbcsp, paper_delta_vs_fbcsp=paper_delta,
            ))
    display(pd.DataFrame(summary).round(1))
else:
    print(f"No results in {RESULTS_DIR} yet")